In [2]:
print('hi')

hi


In [40]:
class VendingMachineException(Exception):
    pass


class InvalidSlotException(VendingMachineException):
    pass


class OutOfStockException(VendingMachineException):
    pass


class InvalidOperationException(VendingMachineException):
    pass


class InsufficientMoneyException(VendingMachineException):
    pass

In [41]:
class Product:
    def __init__(self, name : str, product_id : str, price : float):
        self.name = name
        self.product_id = product_id
        self.price = price


In [42]:
class Slot:
    def __init__(self, slot_id: str, product: Product, quantity: int):
        self.slot_id = slot_id
        self.product = product
        self.quantity = quantity
    
    def is_available(self):
        return self.quantity > 0
    
    def reduce_quantity(self):
        if not self.is_available():
            raise OutOfStockException("Product out of stock")
        
        self.quantity -= 1

In [43]:
class Inventory:
    def __init__(self, inventory_id: str):
        self.inventory_id = inventory_id
        self.all_slots = []
    
    def add_slot(self, slot: Slot):
        for existing_slot in self.all_slots:
            if existing_slot.slot_id == slot.slot_id:
                raise InvalidSlotException("Duplicate slot not allowed")
        
        self.all_slots.append(slot)
    
    def get_slot(self, slot_id: str):
        for slot in self.all_slots:
            if slot.slot_id == slot_id:
                return slot
        
        raise InvalidSlotException("Invalid slot id")
    
    def is_available(self, slot_id: str) -> bool:
        slot = self.get_slot(slot_id)
        return slot.is_available()
    
    def get_all_slots(self):
        return list(self.all_slots)

In [44]:
from abc import ABC , abstractmethod

In [45]:
from abc import ABC, abstractmethod

class VendingMachineState(ABC):
    @abstractmethod
    def select_product(self, machine, slot_id: str):
        pass

    @abstractmethod
    def insert_money(self, machine, amount: int):
        pass

    @abstractmethod
    def dispense_product(self, machine):
        pass

    @abstractmethod
    def cancel(self, machine):
        pass

In [46]:
class PaymentReceivedState(VendingMachineState):
    def select_product(self, machine, slot_id: str):
        raise InvalidOperationException("Cancel current transaction before selecting another product")

    def insert_money(self, machine, amount: int):
        if amount <= 0:
            raise InsufficientMoneyException("Invalid amount")

        machine.inserted_money += amount
        print(f"Total inserted money: {machine.inserted_money}")

    def dispense_product(self, machine):
        price = machine.selected_slot.product.price

        if machine.inserted_money < price:
            raise InsufficientMoneyException("Not enough money. Please insert more money")

        change = machine.inserted_money - price
        product_name = machine.selected_slot.product.name

        machine.selected_slot.reduce_quantity()

        machine.selected_slot = None
        machine.inserted_money = 0
        machine.state = IdleState()

        if change > 0:
            print(f"Dispensed {product_name}. Change returned: {change}")
        else:
            print(f"Dispensed {product_name}")

    def cancel(self, machine):
        refund = machine.inserted_money

        machine.selected_slot = None
        machine.inserted_money = 0
        machine.state = IdleState()

        print(f"Transaction cancelled. Refunded: {refund}")

In [47]:
class ProductSelectedState(VendingMachineState):
    def select_product(self, machine, slot_id: str):
        slot = machine.inventory.get_slot(slot_id)

        if not slot.is_available():
            raise OutOfStockException("Slot is empty")

        machine.selected_slot = slot
        print(f"Changed selected product to: {slot.product.name}")

    def insert_money(self, machine, amount: int):
        if amount <= 0:
            raise Exception("Invalid amount")

        machine.inserted_money += amount
        machine.state = PaymentReceivedState()

        print(f"Total inserted money: {machine.inserted_money}")

    def dispense_product(self, machine):
        raise InvalidOperationException("Please insert money first")

    def cancel(self, machine):
        machine.selected_slot = None
        machine.inserted_money = 0
        machine.state = IdleState()

        print("Transaction cancelled")

In [48]:
class IdleState(VendingMachineState):
    def select_product(self, machine, slot_id: str):
        slot = machine.inventory.get_slot(slot_id)

        if not slot.is_available():
            raise OutOfStockException("Slot is empty")

        machine.selected_slot = slot
        machine.state = ProductSelectedState()

        print(f"Selected product: {slot.product.name}")
    
    def insert_money(self, machine, amount: int):
        raise InvalidOperationException("Please select product first")
    
    def dispense_product(self, machine):
        raise InvalidOperationException("Please select product first")
    
    def cancel(self, machine):
        raise InvalidOperationException("No active transaction")

In [49]:
class VendingMachine:
    def __init__(self, inventory: Inventory):
        self.inventory = inventory
        self.selected_slot = None
        self.inserted_money = 0
        self.state = IdleState()

    def select_product(self, slot_id: str):
        self.state.select_product(self, slot_id)

    def insert_money(self, amount: int):
        self.state.insert_money(self, amount)

    def dispense_product(self):
        self.state.dispense_product(self)

    def cancel(self):
        self.state.cancel(self)

    def display_products(self):
        for slot in self.inventory.get_all_slots():
            print(
                f"Slot = {slot.slot_id}, "
                f"Product = {slot.product.name}, "
                f"Price = {slot.product.price}, "
                f"Quantity = {slot.quantity}"
            )

In [50]:
# ---------- Create Products ----------
coke = Product("Coke", "P101", 40)
chips = Product("Chips", "P102", 20)
water = Product("Water", "P103", 15)


# ---------- Create Slots ----------
slot1 = Slot("A1", coke, 5)
slot2 = Slot("B1", chips, 3)
slot3 = Slot("C1", water, 0)


# ---------- Create Inventory ----------
inventory = Inventory("INV001")
inventory.add_slot(slot1)
inventory.add_slot(slot2)
inventory.add_slot(slot3)


# ---------- Create Vending Machine ----------
machine = VendingMachine(inventory)


print("\n========== TEST 1: Display Products ==========")
machine.display_products()


print("\n========== TEST 2: Successful Purchase ==========")
machine.select_product("A1")
machine.insert_money(40)
machine.dispense_product()
machine.display_products()


print("\n========== TEST 3: Purchase With Change ==========")
machine.select_product("B1")
machine.insert_money(50)
machine.dispense_product()
machine.display_products()


print("\n========== TEST 4: Out Of Stock Exception ==========")
try:
    machine.select_product("C1")
except OutOfStockException as e:
    print("OutOfStockException:", e)


print("\n========== TEST 5: Invalid Slot Exception ==========")
try:
    machine.select_product("Z9")
except InvalidSlotException as e:
    print("InvalidSlotException:", e)


print("\n========== TEST 6: Invalid Operation Exception ==========")
try:
    machine.insert_money(20)
except InvalidOperationException as e:
    print("InvalidOperationException:", e)


print("\n========== TEST 7: Insufficient Money Exception ==========")
try:
    machine.select_product("A1")
    machine.insert_money(10)
    machine.dispense_product()
except InsufficientMoneyException as e:
    print("InsufficientMoneyException:", e)

# reset active transaction
machine.cancel()


print("\n========== TEST 8: Cancel After Money Inserted ==========")
machine.select_product("A1")
machine.insert_money(20)
machine.cancel()


print("\n========== TEST 9: Duplicate Slot Exception ==========")
try:
    duplicate_slot = Slot("A1", Product("Pepsi", "P104", 35), 2)
    inventory.add_slot(duplicate_slot)
except InvalidSlotException as e:
    print("InvalidSlotException:", e)


print("\n========== TEST 10: Final Inventory ==========")
machine.display_products()


========== TEST 1: Display Products ==========
Slot = A1, Product = Coke, Price = 40, Quantity = 5
Slot = B1, Product = Chips, Price = 20, Quantity = 3
Slot = C1, Product = Water, Price = 15, Quantity = 0

========== TEST 2: Successful Purchase ==========
Selected product: Coke
Total inserted money: 40
Dispensed Coke
Slot = A1, Product = Coke, Price = 40, Quantity = 4
Slot = B1, Product = Chips, Price = 20, Quantity = 3
Slot = C1, Product = Water, Price = 15, Quantity = 0

========== TEST 3: Purchase With Change ==========
Selected product: Chips
Total inserted money: 50
Dispensed Chips. Change returned: 30
Slot = A1, Product = Coke, Price = 40, Quantity = 4
Slot = B1, Product = Chips, Price = 20, Quantity = 2
Slot = C1, Product = Water, Price = 15, Quantity = 0

========== TEST 4: Out Of Stock Exception ==========
OutOfStockException: Slot is empty

========== TEST 5: Invalid Slot Exception ==========
InvalidSlotException: Invalid slot id

========== TEST 6: Invalid Operation Excepti